# 04 - Statistical Hypothesis Testing

In Stage 3 the charts *suggested* patterns (for example, students with past failures score lower).
But a pattern in a chart could just be **luck**: a different group of 634 students might not show it.

This notebook asks, for each of the five research questions:

1. **Is the pattern real?** Could it be explained by chance alone? (the **p-value**)
2. **How big is it?** Is the difference large enough to matter? (the **effect size**)

## The idea in plain words

Think of a courtroom. The defendant is *innocent until proven guilty*.

- **Null hypothesis (H0):** the "innocent" starting point. *"There is no difference / no relationship."*
  Example: *"Study time has no link to final grade."*
- **Alternative hypothesis (H1):** what we suspect. *"There is a difference / relationship."*
- **p-value:** *If H0 were true (no real effect), how likely is it that we would see a pattern this strong just by luck?*
  - Small p-value (below 0.05) → the data would be very surprising if H0 were true → **reject H0**. The pattern is "statistically significant".
  - Large p-value → the data is not surprising → **fail to reject H0**. (Like "not guilty": it does not prove innocence, it just means not enough evidence.)
- **Significance level (α = 0.05):** the cut-off, chosen **before** looking at results. It means we accept a 5% risk of a false alarm.
- **Effect size:** a p-value only says whether an effect is *real*, not whether it is *big*. With lots of data even a tiny difference can be "significant". Effect size measures *how big*.

## Choosing the right test

The test depends on what kind of column we compare with `G3`:

| Kind of comparison | Example | Main test | Backup test (rank-based) | Effect size |
|---|---|---|---|---|
| **2 groups** | GP vs MS | Welch's t-test | Mann-Whitney U | Cohen's d |
| **3 or more groups** | study time 1–4 | One-way ANOVA | Kruskal-Wallis | Eta squared (η²) |
| **Number vs number** | absences vs G3 | Pearson correlation | Spearman correlation | r itself |

**Why a backup test?** The main tests work best when grades are roughly bell-shaped (normal). The backup tests make no such assumption: they rank students from lowest to highest grade and compare the ranks instead. If both tests agree, we can trust the result.

## Setup

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

In [2]:
df = pd.read_csv('../data/processed/student-por-clean.csv')
df.shape

(634, 33)

`ALPHA` is our cut-off, decided now, before seeing any results.
`results` is an empty list. After each test we add one row to it, so at the end we can show everything in one table.

In [3]:
ALPHA = 0.05
results = []

## Reading p-values: scientific notation

p-values here are often tiny and Python shows them like `1.09e-09`.
`e-09` means "move the decimal point 9 places to the left": `1.09e-09` = 0.00000000109.
Anything with `e-` followed by 2 or more is far below 0.05.

## Checking the normality assumption

The main tests (t-test, ANOVA, Pearson) assume `G3` is roughly bell-shaped.

- **Skewness** measures lopsidedness. 0 = perfectly symmetric. Between -0.5 and 0.5 counts as roughly symmetric.
- **Shapiro-Wilk test** is itself a hypothesis test. Its H0 is *"the data is normal"*.

In [4]:
print('Skewness:', round(df['G3'].skew(), 3))
print('Shapiro-Wilk p-value:', stats.shapiro(df['G3']).pvalue)

Skewness: 0.125
Shapiro-Wilk p-value: 1.5910616733484623e-08


**Observation:** `G3` is almost perfectly symmetric (skewness ≈ 0.13), and the Stage 3 histogram looked bell-shaped.
Shapiro-Wilk still rejects normality (p is tiny), but that is expected: with 634 students it flags even very small departures, and grades are whole numbers (0–20), which are never perfectly normal.

With groups this large, the t-test and ANOVA still work well. To be safe, every question also gets a backup rank-based test.

## Effect size helpers

These two small functions calculate effect sizes. `scipy` does not provide them directly.

**Cohen's d** (for 2 groups): the difference between the two averages, measured in "standard deviations" (a standard deviation is the typical distance of a grade from the average).
Rough guide: 0.2 = small, 0.5 = medium, 0.8 = large.

**Eta squared, η²** (for 3+ groups): the share of all the variation in grades that is explained by which group a student is in.
η² = 0.07 means "7% of the differences in grades line up with the groups".
Rough guide: 0.01 = small, 0.06 = medium, 0.14 = large.

In [5]:
def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    pooled_sd = np.sqrt(((n1 - 1) * group1.var() + (n2 - 1) * group2.var()) / (n1 + n2 - 2))
    return (group1.mean() - group2.mean()) / pooled_sd


def eta_squared(data, group_col, value_col='G3'):
    overall_mean = data[value_col].mean()
    total_variation = ((data[value_col] - overall_mean) ** 2).sum()
    between_groups = sum(
        len(group) * (group[value_col].mean() - overall_mean) ** 2
        for _, group in data.groupby(group_col)
    )
    return between_groups / total_variation

## Q1. Is study time associated with final grade?

- **H0:** average `G3` is the same in all four study-time groups.
- **H1:** at least one group has a different average.

`studytime` has 4 groups → **one-way ANOVA** (backup: **Kruskal-Wallis**).

ANOVA compares how far apart the group averages are against how spread out students are *inside* each group. Groups far apart + little spread inside = strong evidence of a real difference.

First, split `G3` into one list per study-time group.
Read the line as: *"for each group that `groupby` makes, keep only its `G3` column"*. The `_` holds the group's label (1, 2, 3, 4), which we don't need here.

In [6]:
studytime_groups = [group['G3'] for _, group in df.groupby('studytime')]
[len(g) for g in studytime_groups]

[204, 298, 97, 35]

The `*` in front of the list "unpacks" it, so `f_oneway(*groups)` is the same as `f_oneway(group1, group2, group3, group4)`.

In [7]:
anova = stats.f_oneway(*studytime_groups)
kruskal = stats.kruskal(*studytime_groups)
eta2 = eta_squared(df, 'studytime')

print(f'ANOVA:          F = {anova.statistic:.2f}, p = {anova.pvalue:.2e}')
print(f'Kruskal-Wallis: H = {kruskal.statistic:.2f}, p = {kruskal.pvalue:.2e}')
print(f'Eta squared:    {eta2:.3f}')

results.append({'question': 'Q1 Study time', 'test': 'ANOVA', 'p_value': anova.pvalue,
                'backup_p_value': kruskal.pvalue, 'effect_size': f'η² = {eta2:.3f}'})

ANOVA:          F = 15.42, p = 1.09e-09
Kruskal-Wallis: H = 45.58, p = 6.98e-10
Eta squared:    0.068


**Result:** p is far below 0.05 in both tests → **reject H0**. Study time is significantly associated with final grade.
η² ≈ 0.07 is a **medium** effect: study time lines up with about 7% of the differences in grades. So it matters, but most of the variation comes from other things.

## Q2. Are absences associated with final grade?

- **H0:** there is no correlation between absences and `G3`.
- **H1:** there is a correlation.

Both columns are numbers → **Pearson correlation** (backup: **Spearman**).

- **Pearson** measures how well the points fit a straight line.
- **Spearman** does the same on ranks. It is safer here because `absences` is very skewed (most students have 0–4, a few have 30+), and a few extreme values can pull Pearson around.

The correlation `r` is its own effect size. Rough guide: 0.1 = small, 0.3 = medium, 0.5 = large (ignore the minus sign for size).

In [8]:
pearson = stats.pearsonr(df['absences'], df['G3'])
spearman = stats.spearmanr(df['absences'], df['G3'])

print(f'Pearson:  r = {pearson.statistic:.3f}, p = {pearson.pvalue:.2e}')
print(f'Spearman: r = {spearman.statistic:.3f}, p = {spearman.pvalue:.2e}')
print(f'r squared: {pearson.statistic ** 2:.3f}')

results.append({'question': 'Q2 Absences', 'test': 'Pearson', 'p_value': pearson.pvalue,
                'backup_p_value': spearman.pvalue, 'effect_size': f'r = {pearson.statistic:.3f}'})

Pearson:  r = -0.196, p = 6.54e-07
Spearman: r = -0.214, p = 5.23e-08
r squared: 0.038


**Result:** p is far below 0.05 in both tests → **reject H0**. More absences go with lower grades.
But r ≈ -0.20 is a **small** effect. Squaring it (r² ≈ 0.04) shows absences line up with only about 4% of the differences in grades.

This is a good example of **significant ≠ important**: we are confident the link is real, but it is weak.

## Q3. Are previous failures associated with final grade?

- **H0:** average `G3` is the same for all failure groups (0, 1, 2, 3+).
- **H1:** at least one group has a different average.

4 groups → **ANOVA** (backup: **Kruskal-Wallis**).

The backup test matters most here: the groups have very different sizes (543 vs 15 vs 13) and different spreads, which bends ANOVA's rules.

In [9]:
failures_groups = [group['G3'] for _, group in df.groupby('failures')]
[len(g) for g in failures_groups]

[543, 63, 15, 13]

In [10]:
anova = stats.f_oneway(*failures_groups)
kruskal = stats.kruskal(*failures_groups)
eta2 = eta_squared(df, 'failures')

print(f'ANOVA:          F = {anova.statistic:.2f}, p = {anova.pvalue:.2e}')
print(f'Kruskal-Wallis: H = {kruskal.statistic:.2f}, p = {kruskal.pvalue:.2e}')
print(f'Eta squared:    {eta2:.3f}')

results.append({'question': 'Q3 Previous failures', 'test': 'ANOVA', 'p_value': anova.pvalue,
                'backup_p_value': kruskal.pvalue, 'effect_size': f'η² = {eta2:.3f}'})

ANOVA:          F = 45.13, p = 1.98e-26
Kruskal-Wallis: H = 117.81, p = 2.29e-25
Eta squared:    0.177


**Result:** p is extremely small in both tests → **reject H0**.
η² ≈ 0.18 is a **large** effect, the biggest in this project: past failures line up with about 18% of the differences in final grades.

## Q4. Is parental education associated with final grade?

- **H0:** average `G3` is the same at every level of mother's (or father's) education.
- **H1:** at least one level has a different average.

5 groups each → **ANOVA** (backup: **Kruskal-Wallis**), run once for `Medu` and once for `Fedu`.

The same code is needed twice, so a `for` loop runs it once per column.

In [11]:
for col, label in [('Medu', "Q4 Mother's education"), ('Fedu', "Q4 Father's education")]:
    groups = [group['G3'] for _, group in df.groupby(col)]
    anova = stats.f_oneway(*groups)
    kruskal = stats.kruskal(*groups)
    eta2 = eta_squared(df, col)

    print(label)
    print(f'  ANOVA:          F = {anova.statistic:.2f}, p = {anova.pvalue:.2e}')
    print(f'  Kruskal-Wallis: H = {kruskal.statistic:.2f}, p = {kruskal.pvalue:.2e}')
    print(f'  Eta squared:    {eta2:.3f}')

    results.append({'question': label, 'test': 'ANOVA', 'p_value': anova.pvalue,
                    'backup_p_value': kruskal.pvalue, 'effect_size': f'η² = {eta2:.3f}'})

Q4 Mother's education
  ANOVA:          F = 14.71, p = 1.72e-11
  Kruskal-Wallis: H = 59.31, p = 4.04e-12
  Eta squared:    0.086
Q4 Father's education
  ANOVA:          F = 7.12, p = 1.32e-05
  Kruskal-Wallis: H = 32.06, p = 1.86e-06
  Eta squared:    0.043


**Result:** both are significant → **reject H0** for mother's and father's education.
Mother's education has a **medium** effect (η² ≈ 0.09); father's is **small** (η² ≈ 0.04). This matches the Stage 3 charts.

**Association is not causation.** A significant test does not mean parental education *causes* higher grades.
For example, the two schools have very different mixes of parents:

In [12]:
pd.crosstab(df['school'], df['Medu'], normalize='index').round(2)

Medu,0,1,2,3,4
school,,,,,
GP,0.01,0.14,0.29,0.24,0.32
MS,0.01,0.38,0.27,0.17,0.17


`normalize='index'` turns counts into shares of each row. About 56% of GP mothers have secondary or higher education (levels 3–4), compared with about 34% at MS.
So part of the "parental education" effect could really be a "school" effect, or the reverse. Separating them needs a model that looks at several factors at once (Stage 6).

## Q5. Does final grade differ between the two schools?

- **H0:** GP and MS students have the same average `G3`.
- **H1:** the averages are different.

2 groups → **Welch's t-test** (backup: **Mann-Whitney U**).

`equal_var=False` makes it *Welch's* version, which does not assume both groups have the same spread. It is the safer default, especially when group sizes differ (422 vs 212).

In [13]:
gp = df[df['school'] == 'GP']['G3']
ms = df[df['school'] == 'MS']['G3']

ttest = stats.ttest_ind(gp, ms, equal_var=False)
mannwhitney = stats.mannwhitneyu(gp, ms)
d = cohens_d(gp, ms)

print(f"Welch's t-test: t = {ttest.statistic:.2f}, p = {ttest.pvalue:.2e}")
print(f'Mann-Whitney U: U = {mannwhitney.statistic:.0f}, p = {mannwhitney.pvalue:.2e}')
print(f"Cohen's d:      {d:.2f}")

results.append({'question': 'Q5 School', 'test': "Welch's t-test", 'p_value': ttest.pvalue,
                'backup_p_value': mannwhitney.pvalue, 'effect_size': f'd = {d:.2f}'})

Welch's t-test: t = 5.52, p = 6.26e-08
Mann-Whitney U: U = 58306, p = 3.36e-10
Cohen's d:      0.48


**Result:** p is far below 0.05 in both tests → **reject H0**. GP students score significantly higher.
Cohen's d ≈ 0.48 is a **small-to-medium** effect (just under 0.5): GP students average about half a standard deviation (about 1.2 grade points) higher.

## Running many tests at once

Each test has a 5% chance of a false alarm. Run 6 tests and the chance that *at least one* is a false alarm grows to about 26%.

The simplest fix is the **Bonferroni correction**: divide α by the number of tests. With 6 tests the new cut-off is 0.05 / 6 ≈ 0.0083.

In [14]:
bonferroni_alpha = ALPHA / len(results)
round(bonferroni_alpha, 4)

0.0083

## Summary of all tests

In [15]:
summary = pd.DataFrame(results)
summary['significant'] = (summary['p_value'] < bonferroni_alpha) & (summary['backup_p_value'] < bonferroni_alpha)
summary['p_value'] = summary['p_value'].map('{:.1e}'.format)
summary['backup_p_value'] = summary['backup_p_value'].map('{:.1e}'.format)
summary

,question,test,p_value,backup_p_value,effect_size,significant
0,Q1 Study time,ANOVA,1.1e-09,7.0e-10,η² = 0.068,True
1,Q2 Absences,Pearson,6.5e-07,5.2e-08,r = -0.196,True
2,Q3 Previous failures,ANOVA,2.0e-26,2.3e-25,η² = 0.177,True
3,Q4 Mother's education,ANOVA,1.7e-11,4.0e-12,η² = 0.086,True
4,Q4 Father's education,ANOVA,1.3e-05,1.9e-06,η² = 0.043,True
5,Q5 School,Welch's t-test,6.3e-08,3.4e-10,d = 0.48,True


**Every result stays significant even with the stricter Bonferroni cut-off**, and the main and backup tests always agree.
Ranked by effect size (how much each factor matters):

| Rank | Factor | Effect size | Size |
|---|---|---|---|
| 1 | Previous failures | η² = 0.18 | Large |
| 2 | Mother's education | η² = 0.09 | Medium |
| 3 | Study time | η² = 0.07 | Medium |
| 4 | School (GP vs MS) | d = 0.48 | Small-to-medium |
| 5 | Father's education | η² = 0.04 | Small |
| 6 | Absences | r = -0.20 | Small |

## Conclusions

- All five patterns seen in Stage 3 are **real** (very unlikely to be due to chance).
- They are **not equally important**. Previous failures stand out; absences and father's education are real but weak.
- No single factor explains most of the variation in grades. The largest, failures, accounts for about 18%.

## Limitations

- **Association, not causation.** These tests show factors go together with grades, not that they cause them. Factors overlap (e.g. school and parental education).
- **One factor at a time.** Each test ignores all the other factors. Stage 6 (machine learning) will look at them together.
- **Small groups.** Some groups are tiny (2 and 3+ failures, parents with no education), so their individual averages are less reliable.
- **One dataset.** Students come from two Portuguese schools in one subject, so results may not apply elsewhere.